In [1]:
"""
qPCR copies/mL vs. sequencing RPKMF, ALL DATA, all 5 viruses combined on one
plot -- quad-marginal jointplot (main scatter + KDE violin marginals on all
four sides).

Filter: mismatch_flag_after_correction == "none" only (qPCR and sequencing site/date agree,
per the historical convention).

Layout:
  - Main scatter: ALL samples (no positivity filter), symlog x/y so exact
    zeros sit at the axis origin. Colored + shaped per virus.
  - Top/right marginals: log-KDE of each axis's OWN positive values (all
    samples with that axis > 0, regardless of the other axis), per virus.
    Annotated with that axis's overall exact-zero share.
  - Left marginal: log-KDE of rpkmf among qPCR-negative samples (x == 0),
    per virus -- "if qPCR called this negative, what did sequencing still
    see?" Axis inverted so density grows away from the shared axis.
  - Bottom marginal: log-KDE of copies/mL among sequencing-negative samples
    (y == 0), per virus. Axis inverted (density dips below the shared axis).
  - Stats box: n, Pearson r, Spearman rho on the FULL (zero-inclusive) pooled
    x/y arrays.
"""

#Import required packages
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpecFromSubplotSpec
from scipy.stats import pearsonr, spearmanr

#Set colors for plot
BG = "#fdfdfb"
GRID = "#d9d8d4"
DARK = "#333333"

#Input and output file names
DATA_PATH = "Data_3_Correlation_Data.xlsx"
OUT_PATH = "fig3Ai_copiesperml_vs_rpkmf_allviruses_alldata.png"

VIRUSES = [
    ("fluA", "Influenza A", "#3C83D9", "o"),
    ("fluB", "Influenza B", "#EC7444", "s"),
    ("rsvA", "RSV A", "#2CB483", "^"),
    ("rsvB", "RSV B", "#5748AD", "D"),
    ("sars-cov-2", "SARS-COV-2", "#E45655", "P"),
]

FONT_SCALE = 1.4   # 1.0 = current sizes; raise or lower to taste
def fs(size):
    return size * FONT_SCALE

def load_sheet(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    df = df[df["mismatch_flag_after_correction"] == "none"].copy()
    x = df["copiesperml"].to_numpy(dtype=float)
    y = df["rpkmf"].to_numpy(dtype=float)
    return x, y


def draw_marginal_kde(ax, values, orientation, color):
    """Log-KDE half-violin (no rain) -- matches the historical alldata design."""
    from scipy.stats import gaussian_kde
    values = np.asarray(values, dtype=float)
    logv = np.log10(values)
    kde = gaussian_kde(logv)
    grid = np.linspace(logv.min(), logv.max(), 200)
    dens = kde(grid)
    dens = dens / dens.max()
    lin_grid = 10 ** grid
    if orientation == "x":
        ax.fill_between(lin_grid, 0, dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(lin_grid, dens, color=color, alpha=0.95, linewidth=2)
    else:
        ax.fill_betweenx(lin_grid, 0, dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(dens, lin_grid, color=color, alpha=0.95, linewidth=2)


def style_marginal(ax):
    ax.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
                    labelbottom=False, labeltop=False, labelleft=False, labelright=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor("none")


def auto_linthresh(values):
    pos = values[values > 0]
    if len(pos) == 0:
        return 1.0
    return 10 ** np.floor(np.log10(pos.min()))


def build_quad_panel(fig, spec, series):
    """series: list of (label, color, marker, x, y) -- one entry per virus."""
    nested = GridSpecFromSubplotSpec(
        3, 3, subplot_spec=spec, width_ratios=[1, 4, 1], height_ratios=[1, 4, 1],
        hspace=0.18, wspace=0.18,
    )
    ax_main = fig.add_subplot(nested[1, 1])
    ax_top = fig.add_subplot(nested[0, 1], sharex=ax_main)
    ax_right = fig.add_subplot(nested[1, 2], sharey=ax_main)
    ax_left = fig.add_subplot(nested[1, 0], sharey=ax_main)
    ax_bottom = fig.add_subplot(nested[2, 1], sharex=ax_main)
    for ax in (ax_main, ax_top, ax_right, ax_left, ax_bottom):
        ax.set_facecolor(BG)

    all_x = np.concatenate([s[3] for s in series])
    all_y = np.concatenate([s[4] for s in series])
    linthresh_x = auto_linthresh(all_x)
    linthresh_y = max(auto_linthresh(all_y), 1e-3)
    for label, color, marker, x, y in series:
        ax_main.scatter(x, y, s=50, color=color, alpha=0.5, linewidth=0, marker=marker, zorder=3,
                         label=f"{label} (n={len(x):,})")

        x_pos, y_pos = x[x > 0], y[y > 0]
        if len(x_pos) >= 2:
            draw_marginal_kde(ax_top, x_pos, "x", color)
        if len(y_pos) >= 2:
            draw_marginal_kde(ax_right, y_pos, "y", color)

        y_given_x0 = y[(x == 0) & (y > 0)]
        if len(y_given_x0) >= 2:
            draw_marginal_kde(ax_left, y_given_x0, "y", color)
        x_given_y0 = x[(y == 0) & (x > 0)]
        if len(x_given_y0) >= 2:
            draw_marginal_kde(ax_bottom, x_given_y0, "x", color)

    ax_main.set_xscale("symlog", linthresh=linthresh_x)
    ax_main.set_yscale("symlog", linthresh=linthresh_y)
    ax_main.set_xlim(left=-linthresh_x * 0.6)
    ax_main.set_ylim(bottom=-linthresh_y * 0.6)
    ax_top.set_ylim(0, 1.1)
    ax_right.set_xlim(0, 1.1)
    ax_left.set_xlim(0, 1.1)
    ax_bottom.set_ylim(0, 1.1)
    ax_left.invert_xaxis()
    ax_bottom.invert_yaxis()
    ax_legend = fig.add_subplot(nested[0, 2])
    ax_legend.axis("off")
    
    for ax in (ax_top, ax_right, ax_left, ax_bottom):
        style_marginal(ax)

    ax_main.grid(True, color=GRID, alpha=0.9, linewidth=0.9)
    ax_main.set_axisbelow(True)
    ax_main.tick_params(axis="both", labelsize=fs(14))
    
    n_total = len(all_x)
    r, rp = pearsonr(all_x, all_y)
    rho, rhop = spearmanr(all_x, all_y)
    
    #stats_txt = f"n = {n_total:,}\nPearson r = {r:.3f} (p = {rp:.1e})\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})"
    stats_txt = f"n = {n_total:,}\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})"
    ax_main.text(0.03, 0.97, stats_txt, transform=ax_main.transAxes, va="top", ha="left",
                 fontsize=fs(16), bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cccccc", alpha=1.0),
                 zorder=5)

    x_zero_share = (all_x == 0).mean()
    y_zero_share = (all_y == 0).mean()
   
    n_x0 = (all_x == 0).sum()
    n_y0 = (all_y == 0).sum()
    pct_x0_also_y0 = (all_y[all_x == 0] == 0).mean() if n_x0 else 0
    pct_y0_also_x0 = (all_x[all_y == 0] == 0).mean() if n_y0 else 0
    
    ax_left.text(0.03, 1.02, f"qPCR-negative, sequencing-positive\n (n={n_x0:,})",
                 transform=ax_left.transAxes, ha="left", va="bottom", fontsize=fs(14), color="#555555")
    ax_bottom.text(0.0, -0.06, f"sequencing-negative, qPCR-positive\n (n={n_y0:,})",
                   transform=ax_bottom.transAxes, ha="left", va="top", fontsize=fs(14), color="#555555")

    ax_bottom.text(0.5, -0.22, "Copies / mL (qPCR)", transform=ax_bottom.transAxes,
                   ha="center", va="top", fontsize=fs(18), fontweight="bold", color="black")
    ax_left.text(-0.28, 0.5, "RPKMF (Sequencing)", transform=ax_left.transAxes, rotation=90,
                 ha="right", va="center", fontsize=fs(18), fontweight="bold", color="black")
    
    handles, labels = ax_main.get_legend_handles_labels()
    ax_legend.legend(handles, labels, loc="center", fontsize=fs(16), frameon=True, facecolor="white",
        edgecolor="#cccccc", framealpha=1.0, title="Virus", title_fontsize=fs(18), markerscale=2)

    return ax_main

def main():
    series = []
    for sheet, label, color, marker in VIRUSES:
        x, y = load_sheet(sheet)
        series.append((label, color, marker, x, y))
        print(f"[{label}] n={len(x):,}")

    n_total = sum(len(s[3]) for s in series)

    fig = plt.figure(figsize=(20, 14), facecolor=BG)
    gs = fig.add_gridspec(1, 1, left=0.08, right=0.88, top=0.80, bottom=0.06)
    build_quad_panel(fig, gs[0, 0], series)

    fig.suptitle(f"qPCR copies/mL vs sequencing RPKMF, all viruses combined",
                 fontsize=fs(18), fontweight="bold", ha="center")
 
    fig.savefig(OUT_PATH, dpi=300, facecolor=fig.get_facecolor())
    fig.savefig(OUT_PATH.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"[combined] n={n_total:,} -> {OUT_PATH}")


if __name__ == "__main__":
    main()


[Influenza A] n=3,910
[Influenza B] n=3,897
[RSV A] n=3,879
[RSV B] n=3,914
[SARS-COV-2] n=3,910
[combined] n=19,510 -> fig3Ai_copiesperml_vs_rpkmf_allviruses_alldata.png


In [2]:
"""qPCR copies/mL vs. sequencing RPKMF, DOUBLE-POSITIVE ONLY, all 5 viruses
combined on one plot -- jointplot style (main scatter + top/right marginal
log-KDE density curves + log-log OLS fit line).

Layout:
  - Main scatter: double-positive samples only, log-log axes. Colored +
    shaped per virus.
  - Top/right marginals: log-KDE of each axis's values (all double-positive,
    so no negative/zero handling needed here), per virus.
  - log-log OLS fit line across the pooled double-positive data, with R² in
    the legend and in the stats box.
  - Stats box: n, Pearson r, Spearman rho, log-log OLS slope/R² on the pooled
    double-positive x/y arrays.
  - Virus legend placed in the unused bottom-right corner, outside the main
    scatter/marginal panels (matching the alldata combined script's layout
    preference established this session).

"""

#Import required packages
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpecFromSubplotSpec
from scipy.stats import gaussian_kde, linregress, pearsonr, spearmanr

#Set colors for plot
BG = "#fdfdfb"
GRID = "#d9d8d4"
DARK = "#333333"

#Input and output file names
DATA_PATH = "Data_3_Correlation_Data.xlsx"
OUT_PATH = "fig3Aii_copiesperml_vs_rpkmf_allviruses_positive_only.png"

VIRUSES = [
    ("fluA", "Influenza A", "#3C83D9", "o"),
    ("fluB", "Influenza B", "#EC7444", "s"),
    ("rsvA", "RSV A", "#2CB483", "^"),
    ("rsvB", "RSV B", "#5748AD", "D"),
    ("sars-cov-2", "SARS-COV-2", "#E45655", "P"),
]

FONT_SCALE = 1.4 # 1.0 = current sizes; raise or lower to taste
def fs(size):
    return size * FONT_SCALE
    
def load_sheet(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    df = df[df["mismatch_flag_after_correction"] == "none"]
    df = df[(df["qpcr_detected"] == "detected") & (df["sequencing_detected"] == "detected")]
    x = df["copiesperml"].to_numpy(dtype=float)
    y = df["rpkmf"].to_numpy(dtype=float)
    return x, y


def draw_marginal_kde(ax, values, orientation, color):
    """Log-KDE half-violin (no rain), matching the alldata combined script."""
    values = np.asarray(values, dtype=float)
    logv = np.log10(values)
    kde = gaussian_kde(logv)
    grid = np.linspace(logv.min(), logv.max(), 200)
    dens = kde(grid)
    dens = dens / dens.max()
    lin_grid = 10 ** grid
    if orientation == "x":
        ax.fill_between(lin_grid, 0, dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(lin_grid, dens, color=color, alpha=0.95, linewidth=2)
    else:
        ax.fill_betweenx(lin_grid, 0, dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(dens, lin_grid, color=color, alpha=0.95, linewidth=2)


def style_marginal(ax):
    ax.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
                    labelbottom=False, labeltop=False, labelleft=False, labelright=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor("none")


def fit_line(x, y):
    log_x, log_y = np.log10(x), np.log10(y)
    res = linregress(log_x, log_y)
    log_x_line = np.linspace(log_x.min(), log_x.max(), 200)
    x_line = 10 ** log_x_line
    y_line = 10 ** (res.slope * log_x_line + res.intercept)
    return x_line, y_line, res


def main():
    series = []
    for sheet, label, color, marker in VIRUSES:
        x, y = load_sheet(sheet)
        series.append((label, color, marker, x, y))
        print(f"[{label}] n={len(x):,}")

    all_x = np.concatenate([s[3] for s in series])
    all_y = np.concatenate([s[4] for s in series])
    n_total = len(all_x)

    fig = plt.figure(figsize=(20, 14), facecolor=BG)
    gs = fig.add_gridspec(1, 1, left=0.08, right=0.97, top=0.86, bottom=0.07)
    nested = GridSpecFromSubplotSpec(
        2, 2, subplot_spec=gs[0, 0], width_ratios=[4, 1], height_ratios=[1, 4],
        hspace=0.06, wspace=0.06,
    )
    ax_top = fig.add_subplot(nested[0, 0])
    ax_main = fig.add_subplot(nested[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(nested[1, 1], sharey=ax_main)
    # unused top-right corner of the 2x2 nested grid -- holds the Virus
    # legend fully outside the scatter/marginal panels.
    ax_legend = fig.add_subplot(nested[0, 1])
    ax_legend.axis("off")
    for ax in (ax_main, ax_top, ax_right):
        ax.set_facecolor(BG)

    for label, color, marker, x, y in series:
        ax_main.scatter(x, y, s=50, color=color, alpha=0.55, linewidth=0, marker=marker, zorder=3,
                         label=f"{label} (n={len(x):,})")
        draw_marginal_kde(ax_top, x, "x", color)
        draw_marginal_kde(ax_right, y, "y", color)

    ax_main.set_xscale("log")
    ax_main.set_yscale("log")
    ax_main.tick_params(axis="both", labelsize=fs(16))
    
    x_line, y_line, res = fit_line(all_x, all_y)
    fit_handle, = ax_main.plot(x_line, y_line, color=DARK, linewidth=2.5, linestyle="--", zorder=4,
                                label=f"log-log OLS fit (R²={res.rvalue**2:.3f})")

    ax_top.set_ylim(0, 1.1)
    ax_right.set_xlim(0, 1.1)
    style_marginal(ax_top)
    style_marginal(ax_right)

    #Correlation coefficient calcualtions (Both Pearson's and Spearman's correlation coefficients) 
    r, rp = pearsonr(all_x, all_y)
    rho, rhop = spearmanr(all_x, all_y)
    #stats_txt = (f"n = {n_total:,}\nPearson r = {r:.3f} (p = {rp:.1e})\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})\n"
     #            f"log-log OLS: slope = {res.slope:.3f}, R² = {res.rvalue**2:.3f}")
    stats_txt = (f"n = {n_total:,}\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})\n"
                 f"log-log OLS: slope = {res.slope:.3f}, R² = {res.rvalue**2:.3f}")
    ax_main.text(0.03, 0.97, stats_txt, transform=ax_main.transAxes, va="top", ha="left",
                 fontsize=fs(16), bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cccccc"))

    ax_main.set_xlabel("Copies / mL (qPCR)", fontsize=fs(18), fontweight="bold")
    ax_main.set_ylabel("RPKMF (Sequencing)", fontsize=fs(18), fontweight="bold")
    ax_main.grid(True, color=GRID, alpha=0.9, linewidth=0.9)
    ax_main.set_axisbelow(True)

    handles, labels = ax_main.get_legend_handles_labels()
    ax_legend.legend(handles, labels, loc="center", fontsize=fs(16), frameon=True, facecolor="white",
        edgecolor="#cccccc", framealpha=1.0, title="Virus", title_fontsize=fs(18), markerscale=2)

    fig.suptitle("qPCR copies/mL vs sequencing RPKMF for all viruses combined — double-positive only",
                 fontsize=fs(18), fontweight="bold",ha="center")
   
    fig.savefig(OUT_PATH, dpi=300, facecolor=fig.get_facecolor())
    fig.savefig(OUT_PATH.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)

    plt.close(fig)
    print(f"[combined] n={n_total:,} -> {OUT_PATH}")


if __name__ == "__main__":
    main()


[Influenza A] n=264
[Influenza B] n=99
[RSV A] n=495
[RSV B] n=169
[SARS-COV-2] n=3,060
[combined] n=4,087 -> fig3Aii_copiesperml_vs_rpkmf_allviruses_positive_only.png
